<h3>CONFIGURATION </h3>

In [2]:
import os
import numpy as np
import librosa
import pandas as pd
from tqdm import tqdm

# ========== CONFIGURATION ==========
DATASET_PATH = "D:\\Music_Gener_Classification\\Data\\genres_original"
OUTPUT_PATH = "processed_data"

SAMPLE_RATE = 22050
SEGMENT_DURATION = 3  # seconds
OVERLAP = 0.5  # 50% overlap

N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

In [3]:
from sklearn.model_selection import train_test_split

def collect_songs(dataset_path):
    songs = []
    
    for genre in os.listdir(dataset_path):
        genre_path = os.path.join(dataset_path, genre)
        
        if os.path.isdir(genre_path):
            for file in os.listdir(genre_path):
                if file.endswith(".wav"):
                    songs.append({
                        "path": os.path.join(genre_path, file),
                        "genre": genre
                    })
    
    return pd.DataFrame(songs)

In [4]:
def split_songs(df):
    train_df, temp_df = train_test_split(
        df,
        test_size=0.3,
        stratify=df["genre"],
        random_state=42
    )
    
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df["genre"],
        random_state=42
    )
    
    return train_df, val_df, test_df

In [5]:
if __name__ == "__main__":
    df = collect_songs(DATASET_PATH)
    train_df, val_df, test_df = split_songs(df)

    print("Total Songs:", len(df))
    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))

Total Songs: 999
Train: 699
Validation: 150
Test: 150


In [7]:
def create_folders():
    for split in ["train", "val", "test"]:
        split_path = os.path.join(OUTPUT_PATH, split)
        os.makedirs(split_path, exist_ok=True)

In [12]:
def process_split(df, split_name):
    split_path = os.path.join(OUTPUT_PATH, split_name)

    segment_samples = int(SAMPLE_RATE * SEGMENT_DURATION)
    hop_samples = int(segment_samples * (1 - OVERLAP))

    for index, row in tqdm(df.iterrows(), total=len(df)):
        file_path = row["path"]
        genre = row["genre"]

        # Load audio
        signal, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)

        # Slide window
        for start in range(0, len(signal) - segment_samples, hop_samples):
            end = start + segment_samples
            segment = signal[start:end]

            # Mel Spectrogram
            mel = librosa.feature.melspectrogram(
                y=segment,
                sr=sr,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH,
                n_mels=N_MELS
            )

            # Convert to dB
            mel_db = librosa.power_to_db(mel, ref=np.max)

            # Save
            filename = f"{genre}_{index}_{start}.npy"
            save_path = os.path.join(split_path, filename)

            np.save(save_path, mel_db)

In [13]:
if __name__ == "__main__":
    create_folders()

    df = collect_songs(DATASET_PATH)
    train_df, val_df, test_df = split_songs(df)

    process_split(train_df, "train")
    process_split(val_df, "val")
    process_split(test_df, "test")

100%|██████████| 150/150 [00:16<00:00,  9.30it/s]


In [11]:
sample = np.load("D:\\Music_Gener_Classification\\processed_data\\train\\blues_0_132300.npy")
print(sample.shape)

(128, 130)
